In [ ]:
import os
csv_dir = "/Users/jehadabed/Desktop/Coding/OCx25-dashboard/v4/Data/"
print(os.listdir(csv_dir))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

main_df = pd.read_csv(f"ExpDataDump_250923.csv")
main_df = main_df.sort_values(by='voltage', ascending=True).reset_index(drop=True)
# Remove rows where 'rep' column contains 'avg' or 'std' (case-insensitive)
if "rep" in main_df.columns:
    main_df = main_df[~main_df["rep"].astype(str).str.lower().str.contains("avg|std", na=False)].reset_index(drop=True)

# Rename "composition" to "target composition" in main_df if it exists
if "composition" in main_df.columns:
    main_df = main_df.rename(columns={"composition": "target composition"})


# Sort by 'source', then 'batch number', then 'batch date' (all ascending), then by 'sample id'
sort_cols = []
if "source" in main_df.columns:
    sort_cols.append("source")
if "batch number" in main_df.columns:
    sort_cols.append("batch number")
if "batch date" in main_df.columns:
    sort_cols.append("batch date")
if "sample id" in main_df.columns:
    sort_cols.append("sample id")
if sort_cols:
    main_df = main_df.sort_values(by=sort_cols, ascending=True).reset_index(drop=True)


main_df.head(10)


In [ ]:
# Load the new XRF data
xrf_df = pd.read_csv(f"{csv_dir}XRFDataDump-250923.csv")
# Sort by 'source', then 'batch number', then 'batch date' (all ascending), then by 'sample id'
sort_cols = []
if "source" in xrf_df.columns:
    sort_cols.append("source")
if "batch number" in xrf_df.columns:
    sort_cols.append("batch number")
if "batch date" in xrf_df.columns:
    sort_cols.append("batch date")
if "sample id" in xrf_df.columns:
    sort_cols.append("sample id")
if sort_cols:
    xrf_df = xrf_df.sort_values(by=sort_cols, ascending=True).reset_index(drop=True)

xrf_df.head(10)

In [ ]:
# For "uoft" samples, ensure all sample ids with the same prefix up to (but not including) "_rep" share the same xrf composition
import re

def get_xrf_composition(row):
    sid = row["sample id"]
    source = row.get("source", None)
    if source == "uoft":
        # Extract the prefix up to (but not including) "_rep"
        match_prefix = re.match(r"(.+_rep)\d+$", sid)
        if match_prefix:
            prefix = match_prefix.group(1)
            # Find all sample ids in xrf_df that start with this prefix
            matches = xrf_df[xrf_df["sample id"].str.startswith(prefix)]
            # Use the first valid xrf-gde_formula or fallback to xrf-wafer_formula
            for _, mrow in matches.iterrows():
                val = mrow.get("xrf-gde_formula", None)
                if pd.notna(val) and str(val).strip() != "":
                    return val
                val2 = mrow.get("xrf-wafer_formula", None)
                if pd.notna(val2) and str(val2).strip() != "":
                    return val2
            return None
    # Default behavior for non-uoft or if no prefix match
    match = xrf_df[xrf_df["sample id"] == sid]
    if not match.empty:
        val = match.iloc[0].get("xrf-gde_formula", None)
        if pd.notna(val) and str(val).strip() != "":
            return val
        val2 = match.iloc[0].get("xrf-wafer_formula", None)
        if pd.notna(val2) and str(val2).strip() != "":
            return val2
    return None

# Insert "xrf composition" column after "target composition"
cols = list(main_df.columns)
if "target composition" in cols:
    insert_at = cols.index("target composition") + 1
else:
    # If not present, add at the end
    insert_at = len(cols)

# First, compute the xrf composition as before
xrf_comps = main_df.apply(get_xrf_composition, axis=1)

# For samples where "xrf composition" is empty, use "target composition" value
def fill_xrf_with_target(xrf_val, target_val):
    if pd.isna(xrf_val) or str(xrf_val).strip() == "":
        return target_val
    return xrf_val

if "target composition" in main_df.columns:
    target_comps = main_df["target composition"]
else:
    target_comps = [None] * len(main_df)

xrf_comps_filled = [
    fill_xrf_with_target(xrf_val, target_val)
    for xrf_val, target_val in zip(xrf_comps, target_comps)
]

main_df.insert(insert_at, "xrf composition", xrf_comps_filled)
main_df.head(10)


In [ ]:
# Decompose "xrf composition" into separate columns for each element

import numpy as np

def parse_xrf_composition(comp):
    """
    Parse a string like 'Ag-0.129-Cu-0.083-Pd-0.788' into a dict: {'Ag': 0.129, 'Cu': 0.083, 'Pd': 0.788}
    """
    if not isinstance(comp, str) or comp.strip() == "":
        return {}
    parts = comp.split('-')
    # Expect alternating element, value, element, value, ...
    out = {}
    i = 0
    while i < len(parts) - 1:
        el = parts[i]
        try:
            val = float(parts[i+1])
        except Exception:
            val = 0.0
        out[el] = val
        i += 2
    return out

# Apply parsing to all rows and collect all unique elements
xrf_dicts = main_df["xrf composition"].apply(parse_xrf_composition)
all_elements = set()
for d in xrf_dicts:
    all_elements.update(d.keys())

# For each element, create a new column, fill missing with 0, and sort columns alphabetically
element_columns = []
for el in sorted(all_elements):
    main_df[el] = xrf_dicts.apply(lambda d: d.get(el, 0.0))
    element_columns.append(el)

# Move the element columns to be together and in alphabetical order (if not already)
# Find the first element column's index
first_el_idx = min([main_df.columns.get_loc(el) for el in element_columns])
# Remove element columns from their current positions
cols = [col for col in main_df.columns if col not in element_columns]
# Insert element columns in sorted order at the correct position
for i, el in enumerate(sorted(element_columns)):
    cols.insert(first_el_idx + i, el)
main_df = main_df[cols]

main_df

In [ ]:
# Fill any column after "current density" with 0 if nan
if "current density" in main_df.columns:
    idx = main_df.columns.get_loc("current density")
    cols_after_cd = main_df.columns[(idx+1):]
    main_df[cols_after_cd] = main_df[cols_after_cd].fillna(0)


In [ ]:
# Export main_df as a CSV file
main_df.to_csv("DashboardData-250923.csv", index=False)
